# PS-Q1 Biomedical Signal Classifier
**Qiskit Fall Fest 2026 – Phase 1**

Hybrid VQC vs Logistic Regression for Normal/Abnormal biomedical signal classification, with Processor A/B benchmarking.

## 1. Environment & Imports

In [ ]:
import sys
sys.path.insert(0, ".")
from src.config import *
from src.utils import set_seed, get_logger
from src.data_loader import load_dataset, save_processed
from src.preprocessing import preprocess_pipeline
from src.feature_extraction import validate_feature_matrix
from src.classical_model import train_logistic_regression, evaluate_classical, save_classical_model
from src.quantum_model import create_vqc, train_vqc, evaluate_vqc, save_vqc_parameters, get_logical_circuit
from src.metrics import metrics_to_row
from src.processors import load_processor
from src.benchmark import measure_circuit_resources
from src.visualization import *
import pandas as pd
import numpy as np
set_seed(RANDOM_SEED)
ensure_dirs()
logger = get_logger("notebook")
print("Environment ready. Seed =", RANDOM_SEED)


## 2. Dataset Loading (Synthetic)

In [ ]:
df, y, feature_names = load_dataset(use_synthetic=True, n_samples=N_SAMPLES)
X = df[feature_names].values
validate_feature_matrix(X, expected_dim=len(feature_names))
print("Samples:", len(df), "| Features:", feature_names)
print(df.head())
print("Class balance:\n", df["label"].value_counts())


## 3. Preprocessing & Split

In [ ]:
X_train, X_test, y_train, y_test, scaler = preprocess_pipeline(X, y)
save_processed(X_train, X_test, y_train, y_test, feature_names)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Scaled range [train]:", X_train.min(), "→", X_train.max())


## 4. Classical Baseline (Logistic Regression)

In [ ]:
clf = train_logistic_regression(X_train, y_train)
y_pred_clf, metrics_clf = evaluate_classical(clf, X_test, y_test)
save_classical_model(clf, metrics_clf)
print(metrics_clf)


## 5. Variational Quantum Classifier

In [ ]:
vqc = create_vqc(n_qubits=N_QUBITS, max_iter=MAX_ITER)
vqc, loss_history = train_vqc(vqc, X_train, y_train)
y_pred_vqc, metrics_vqc = evaluate_vqc(vqc, X_test, y_test)
save_vqc_parameters(vqc)
print(metrics_vqc)


## 6. Logical Circuit & Processor A/B Transpilation

In [ ]:
sample = X_test[0]
logical_qc = get_logical_circuit(vqc, sample)
print("Logical qubits:", logical_qc.num_qubits, "depth:", logical_qc.depth())

res_A = measure_circuit_resources(logical_qc, "A")
res_B = measure_circuit_resources(logical_qc, "B")
print("Processor A:", res_A)
print("Processor B:", res_B)


## 7. Coupling Maps & Resource Figures

In [ ]:
cfg_A = load_processor("A")
cfg_B = load_processor("B")
plot_coupling_map(cfg_A["coupling_map"], cfg_A["physical_qubit_budget"],
                  "Processor A Coupling Map", "processor_A_coupling.png")
plot_coupling_map(cfg_B["coupling_map"], cfg_B["physical_qubit_budget"],
                  "Processor B Coupling Map", "processor_B_coupling.png")
plot_resource_comparison({"Processor A": res_A, "Processor B": res_B},
                         "depth", "Circuit Depth", "circuit_depth_comparison.png")
plot_resource_comparison({"Processor A": res_A, "Processor B": res_B},
                         "cx_count", "CX Count", "cx_count_comparison.png")
plot_resource_comparison({"Processor A": res_A, "Processor B": res_B},
                         "swap_count", "SWAP Count", "swap_count_comparison.png")


## 8. Metrics, Confusion Matrices & Comparison

In [ ]:
plot_confusion_matrix(metrics_clf["confusion_matrix"], "Classical LR", "cm_classical.png")
plot_confusion_matrix(metrics_vqc["confusion_matrix"], "VQC ideal", "cm_vqc.png")
plot_metrics_comparison({"LogisticRegression": metrics_clf, "VQC_ideal": metrics_vqc})
print("Primary metric (Abnormal recall)")
print("  Classical:", metrics_clf["recall_abnormal"])
print("  VQC:      ", metrics_vqc["recall_abnormal"])


## 9. Save Tables & Predictions

In [ ]:
rows = [metrics_to_row("LogisticRegression", metrics_clf),
        metrics_to_row("VQC_ideal", metrics_vqc)]
pd.DataFrame(rows).to_csv(TABLES_DIR / "classification_metrics.csv", index=False)
pd.DataFrame([res_A, res_B]).to_csv(TABLES_DIR / "circuit_resources.csv", index=False)
pd.DataFrame({"y_true": y_test, "y_pred_classical": y_pred_clf, "y_pred_vqc": y_pred_vqc}).to_csv(
    PREDICTIONS_DIR / "test_predictions.csv", index=False)
print("Tables & predictions written to results/")


## 10. Limitations & Conclusions

- Synthetic data only – no clinical claims.
- VQC under-performs classical LR on this linearly separable synthetic task (expected with limited ansatz depth / iterations).
- Compact 4-qubit logical circuit incurs zero SWAP overhead on both Processor A and B under the chosen transpiler settings.
- All seeds fixed; full reproducibility instructions in docs/reproducibility.md.
